# Correlated Predictors Robustness

Addresses the first-pass concerns: event-date conventions with next-trading-day alignment, signed vs absolute EPS revision news, windows that include/exclude the event day, standardized magnitudes, firm/predictor/two-way clustered standard errors, extensive/intensive margins, and a pooled FF12 sector-predictor relevance design.

In [1]:

from pathlib import Path
import json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import h5py
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_cluster, cov_cluster_2groups

ROOT=Path.cwd(); CLEAN=ROOT/'Data'/'clean_data'; RAW=ROOT/'Data'/'data_raw'
RESULTS=ROOT/'Results'/'Estimation'/'Cross_Sectional'; OUT=RESULTS/'correlated_predictors'
OUT.mkdir(parents=True, exist_ok=True)
IBES_PATH=CLEAN/'ibes_detail_eps_forecasts_2010_2017.csv'
FEATURE_PATH=CLEAN/'final_macro_topic_features.csv'
BETAS_PATH=RESULTS/'betas.h5'
PRICE_CACHE=CLEAN/'crsp_daily_prices_2010_2017_sample.csv'
ROBUST_DIR=CLEAN/'correlated_predictors_robustness'
ROBUST_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_CSV=OUT/'correlated_predictors_robustness_table.csv'
MARGIN_CSV=OUT/'correlated_predictors_margin_tests.csv'
SECTOR_CSV=OUT/'correlated_predictors_ff12_sector_table.csv'
SUMMARY_JSON=OUT/'correlated_predictors_robustness_summary.json'
MIN_EVENTS_FIRM=20
MIN_EVENTS_SECTOR=100
print(ROOT)


C:\Users\jonat\Lasso_paper


## Load Selection Rates, Prices, Predictors

In [2]:

with h5py.File(BETAS_PATH,'r') as f:
    stocks=np.array([int(x.decode() if isinstance(x,bytes) else x) for x in f['stocks'][:]])
    topics=np.array([x.decode() if isinstance(x,bytes) else str(x) for x in f['topics'][:]])
    betas=f['betas'][:]
valid=np.isfinite(betas); selected=valid & (betas!=0)
sel_rate=np.divide(selected.sum(axis=2), valid.sum(axis=2), out=np.full(selected.shape[:2],np.nan), where=valid.sum(axis=2)>0)
selection_df=pd.DataFrame(sel_rate,index=stocks,columns=topics).stack(dropna=False).reset_index()
selection_df.columns=['permno','predictor','SelectionRate']
selection_df=selection_df.dropna(subset=['SelectionRate'])
selection_df['AnySelected']=(selection_df['SelectionRate']>0).astype(float)
mean_selection=selection_df['SelectionRate'].mean()

prices=pd.read_csv(PRICE_CACHE,parse_dates=['date']).dropna(subset=['price'])
prices['permno']=prices['permno'].astype(int)
prices=prices.sort_values(['permno','date'])
prices['lag_price']=prices.groupby('permno')['price'].shift(1)
prices=prices.dropna(subset=['lag_price'])[['permno','date','lag_price']]

features=pd.read_csv(FEATURE_PATH,parse_dates=['date']).set_index('date').sort_index()
features=features.rename(columns={col: col.replace(' ','_') for col in features.columns})
missing=[t for t in topics if t not in features.columns]
if missing:
    raise ValueError(f'Missing predictors: {missing[:10]} n={len(missing)}')
features=features.reindex(columns=topics).astype(float)
print(f'Selection rows: {len(selection_df):,}; mean selection={mean_selection:.5f}')
print(f'Prices: {len(prices):,}; feature dates: {len(features):,}; predictors={len(topics)}')


Selection rows: 246,240; mean selection=0.01950
Prices: 2,607,527; feature dates: 1,879; predictors=190


## Build Analyst-Level Quarterly Revision Base

In [3]:

usecols=['permno','analys','estimator','fpedats','anndats','actdats','value','measure','forecast_period_type']
ibes=pd.read_csv(IBES_PATH,usecols=usecols,parse_dates=['fpedats','anndats','actdats'])
ibes=ibes[(ibes['measure'].eq('EPS')) & (ibes['forecast_period_type'].eq('Q')) & (ibes['permno'].isin(stocks))].copy()
ibes=ibes.dropna(subset=['permno','analys','estimator','fpedats','anndats','value'])
ibes['permno']=ibes['permno'].astype(int)
ibes['analyst_id']=ibes['estimator'].astype('Int64').astype(str)+'_'+ibes['analys'].astype('Int64').astype(str)
ibes=ibes.sort_values(['permno','analyst_id','fpedats','anndats','actdats'],kind='mergesort')
ibes['Revision']=ibes.groupby(['permno','analyst_id','fpedats'])['value'].diff()
rev_base=ibes.dropna(subset=['Revision']).copy()
rev_base['min_anndats_actdats']=rev_base[['anndats','actdats']].min(axis=1)
print(f'Quarterly forecast rows: {len(ibes):,}; analyst revision rows: {len(rev_base):,}; firms={rev_base.permno.nunique():,}')


Quarterly forecast rows: 1,327,388; analyst revision rows: 979,875; firms=979


## Helpers

In [4]:

price_groups={p: g[['date','lag_price']].sort_values('date') for p,g in prices.groupby('permno')}

def align_next_trading_day(df, raw_date_col):
    parts=[]
    for permno,g in df.dropna(subset=[raw_date_col]).groupby('permno',sort=False):
        pg=price_groups.get(permno)
        if pg is None or pg.empty:
            continue
        trade_dates=pg['date'].to_numpy(dtype='datetime64[ns]')
        lagp=pg['lag_price'].to_numpy(float)
        raw_dates=g[raw_date_col].to_numpy(dtype='datetime64[ns]')
        idx=np.searchsorted(trade_dates, raw_dates, side='left')
        ok=idx < len(trade_dates)
        if not ok.any():
            continue
        out=g.loc[ok].copy()
        out['date']=pd.to_datetime(trade_dates[idx[ok]])
        out['lag_price']=lagp[idx[ok]]
        parts.append(out)
    return pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()

def build_fund_news(date_col):
    cache=ROBUST_DIR/f'fund_news_{date_col}_next_trading.csv'
    if cache.exists():
        return pd.read_csv(cache,parse_dates=['date'])
    aligned=align_next_trading_day(rev_base,date_col)
    aligned=aligned[(aligned['lag_price'].notna()) & (aligned['lag_price']>0)].copy()
    aligned['ScaledRevision']=aligned['Revision']/aligned['lag_price']
    fund=(aligned.groupby(['permno','date'],as_index=False)
          .agg(FundNews=('ScaledRevision','mean'),
               AbsFundNews=('ScaledRevision',lambda x: np.mean(np.abs(x))),
               NRevisions=('ScaledRevision','size'),
               NAnalysts=('analyst_id','nunique')))
    fund.to_csv(cache,index=False)
    return fund

def exposure_matrix(h=5, include_event_day=False):
    if include_event_day:
        # [ -h, 0 ]: event day plus h prior trading days.
        return features.rolling(h+1,min_periods=h+1).sum()
    # [ -h, -1 ]: h prior trading days, excluding event day.
    return features.shift(1).rolling(h,min_periods=h).sum()

def fundrel_by_group(fund, outcome, h, include_event_day, group_col='permno', min_events=20):
    suffix='incl0' if include_event_day else 'excl0'
    cache=ROBUST_DIR/f'fundrel_{group_col}_{outcome}_h{h}_{suffix}_{fund.attrs.get("date_name","date")}.csv'
    if cache.exists():
        return pd.read_csv(cache)
    expo=exposure_matrix(h,include_event_day).reset_index()
    ev=fund.merge(expo,on='date',how='inner').dropna(subset=list(topics))
    rows=[]; xcols=list(topics)
    for gid,g in ev.groupby(group_col,sort=False):
        n=len(g)
        if n < min_events:
            continue
        y=g[outcome].to_numpy(float); X=g[xcols].to_numpy(float)
        y=y-y.mean(); X=X-X.mean(axis=0)
        ssx=np.sum(X*X,axis=0); ok=ssx>1e-12
        beta=np.full(X.shape[1],np.nan); tval=np.full(X.shape[1],np.nan)
        if ok.any() and n>2:
            beta[ok]=(X[:,ok].T@y)/ssx[ok]
            resid=y[:,None]-X[:,ok]*beta[ok]
            sigma2=np.sum(resid*resid,axis=0)/max(n-2,1)
            se=np.sqrt(sigma2/ssx[ok]); tval[ok]=beta[ok]/se
        rows.append(pd.DataFrame({group_col:gid,'predictor':xcols,'theta':beta,'t_stat':tval,'FundRel':np.abs(tval),'n_events':n}))
    fr=pd.concat(rows,ignore_index=True) if rows else pd.DataFrame(columns=[group_col,'predictor','theta','t_stat','FundRel','n_events'])
    if len(fr):
        lo,hi=fr['FundRel'].quantile([0.01,0.99]); fr['FundRel_w']=fr['FundRel'].clip(lo,hi)
    fr.to_csv(cache,index=False)
    return fr

def twoway_demean(df,col,fe1='permno',fe2='predictor'):
    return df[col]-df.groupby(fe1)[col].transform('mean')-df.groupby(fe2)[col].transform('mean')+df[col].mean()

def second_stage(panel,ycol='SelectionRate', xcol='FundRel_w', intensive=False):
    df=panel.dropna(subset=[ycol,xcol]).copy()
    if intensive:
        df=df[df['SelectionRate']>0].copy()
    df['y_dm']=twoway_demean(df,ycol); df['x_dm']=twoway_demean(df,xcol)
    mod=sm.OLS(df['y_dm'],df[['x_dm']]).fit()
    firm_codes=pd.factorize(df['permno'])[0]
    pred_codes=pd.factorize(df['predictor'])[0]
    cov_f=cov_cluster(mod,firm_codes)
    cov_p=cov_cluster(mod,pred_codes)
    cov_tw=cov_cluster_2groups(mod,firm_codes,pred_codes)[0]
    beta=float(mod.params['x_dm'])
    out={'beta':beta,
         'se_firm':float(np.sqrt(cov_f[0,0])),'se_predictor':float(np.sqrt(cov_p[0,0])),'se_twoway':float(np.sqrt(cov_tw[0,0])),
         'n_obs':int(len(df)),'n_firms':int(df.permno.nunique()),'n_predictors':int(df.predictor.nunique()),
         'mean_y':float(df[ycol].mean()),'sd_y':float(df[ycol].std()),'sd_x':float(df[xcol].std())}
    out['t_twoway']=out['beta']/out['se_twoway'] if out['se_twoway'] else np.nan
    out['std_effect']=out['beta']*out['sd_x']/out['sd_y'] if out['sd_y'] else np.nan
    out['effect_pct_mean']=out['beta']/out['mean_y'] if out['mean_y'] else np.nan
    return out

def ff12_from_sic(sic):
    if pd.isna(sic): return np.nan
    sic=int(sic)
    if 100<=sic<=999 or 2000<=sic<=2399 or 2700<=sic<=2749 or 2770<=sic<=2799 or 3100<=sic<=3199 or 3940<=sic<=3989: return 'Consumer_Nondurables'
    if 2500<=sic<=2519 or 2590<=sic<=2599 or 3630<=sic<=3659 or 3710<=sic<=3711 or sic==3714 or sic==3716 or 3750<=sic<=3751 or sic==3792 or 3900<=sic<=3939 or 3990<=sic<=3999: return 'Consumer_Durables'
    if 2520<=sic<=2589 or 2600<=sic<=2699 or 2750<=sic<=2769 or 3000<=sic<=3099 or 3200<=sic<=3569 or 3580<=sic<=3629 or 3700<=sic<=3709 or 3712<=sic<=3713 or sic==3715 or 3717<=sic<=3749 or 3752<=sic<=3791 or 3793<=sic<=3799 or 3830<=sic<=3839 or 3860<=sic<=3899: return 'Manufacturing'
    if 1200<=sic<=1399 or 2900<=sic<=2999: return 'Energy'
    if 2800<=sic<=2829 or 2840<=sic<=2899: return 'Chemicals'
    if 3570<=sic<=3579 or 3660<=sic<=3692 or 3694<=sic<=3699 or 3810<=sic<=3829 or 7370<=sic<=7379: return 'Business_Equipment'
    if 4800<=sic<=4899: return 'Telecom'
    if 4900<=sic<=4949: return 'Utilities'
    if 5000<=sic<=5999 or 7200<=sic<=7299 or 7600<=sic<=7699: return 'Shops'
    if 2830<=sic<=2839 or 3693<=sic<=3693 or 3840<=sic<=3859 or 8000<=sic<=8099: return 'Health'
    if 6000<=sic<=6999: return 'Finance'
    return 'Other'


## Firm-Level Robustness Table

In [5]:

date_specs=[('ANNDATS_next','anndats'),('ACTDATS_next','actdats'),('MIN_ANN_ACT_next','min_anndats_actdats')]
robust_specs=[]
# Compact requested table, plus date-convention variants for the baseline window.
for label,date_col in date_specs:
    robust_specs.append({'date_label':label,'date_col':date_col,'outcome':'FundNews','h':5,'include_event_day':False})
    robust_specs.append({'date_label':label,'date_col':date_col,'outcome':'AbsFundNews','h':5,'include_event_day':False})
# Extra timing/window robustness on preferred public-announcement convention.
for outcome in ['FundNews','AbsFundNews']:
    for h,inc in [(5,True),(10,False),(20,False),(1,False)]:
        robust_specs.append({'date_label':'ANNDATS_next','date_col':'anndats','outcome':outcome,'h':h,'include_event_day':inc})

rows=[]; margin_rows=[]
for spec in robust_specs:
    fund=build_fund_news(spec['date_col']); fund.attrs['date_name']=spec['date_label']
    fr=fundrel_by_group(fund,spec['outcome'],spec['h'],spec['include_event_day'],'permno',MIN_EVENTS_FIRM)
    panel=selection_df.merge(fr,on=['permno','predictor'],how='inner').dropna(subset=['FundRel_w'])
    res=second_stage(panel,'SelectionRate')
    row={**spec,**res,'fund_rows':len(fund),'fund_firms':fund.permno.nunique(),'fundrel_rows':len(fr)}
    row['window']='[-%d,0]'%spec['h'] if spec['include_event_day'] else '[-%d,-1]'%spec['h']
    rows.append(row)
    if spec['date_label']=='ANNDATS_next' and spec['h']==5 and spec['include_event_day'] is False:
        for ycol,intensive,name in [('AnySelected',False,'extensive_any_selected'),('SelectionRate',True,'intensive_positive_selection')]:
            m=second_stage(panel,ycol,intensive=intensive)
            margin_rows.append({**spec,**m,'margin':name,'window':row['window']})
    print(f"done {spec['date_label']} {spec['outcome']} {row['window']}: beta={res['beta']:.6g}, t2w={res['t_twoway']:.2f}, N={res['n_obs']:,}")
robust=pd.DataFrame(rows)
robust.to_csv(SUMMARY_CSV,index=False)
margins=pd.DataFrame(margin_rows); margins.to_csv(MARGIN_CSV,index=False)
robust[['date_label','outcome','window','beta','se_twoway','t_twoway','std_effect','effect_pct_mean','n_obs','n_firms']]


done ANNDATS_next FundNews [-5,-1]: beta=0.000309815, t2w=2.27, N=175,560
done ANNDATS_next AbsFundNews [-5,-1]: beta=0.000130606, t2w=1.04, N=175,560
done ACTDATS_next FundNews [-5,-1]: beta=0.000193177, t2w=1.48, N=175,750
done ACTDATS_next AbsFundNews [-5,-1]: beta=0.000145013, t2w=1.08, N=175,750
done MIN_ANN_ACT_next FundNews [-5,-1]: beta=0.000310096, t2w=2.27, N=175,560
done MIN_ANN_ACT_next AbsFundNews [-5,-1]: beta=0.00013054, t2w=1.04, N=175,560
done ANNDATS_next FundNews [-5,0]: beta=0.000346439, t2w=2.79, N=175,560
done ANNDATS_next FundNews [-10,-1]: beta=0.000248975, t2w=2.15, N=175,560
done ANNDATS_next FundNews [-20,-1]: beta=0.000218607, t2w=1.90, N=175,560
done ANNDATS_next FundNews [-1,-1]: beta=3.817e-06, t2w=0.03, N=175,560
done ANNDATS_next AbsFundNews [-5,0]: beta=7.53546e-05, t2w=0.60, N=175,560
done ANNDATS_next AbsFundNews [-10,-1]: beta=8.71857e-06, t2w=0.06, N=175,560
done ANNDATS_next AbsFundNews [-20,-1]: beta=0.000120927, t2w=0.95, N=175,560
done ANNDATS_

,date_label,outcome,window,beta,se_twoway,t_twoway,std_effect,effect_pct_mean,n_obs,n_firms
0,ANNDATS_next,FundNews,"[-5,-1]",0.000310,0.000137,2.265996,0.004778,0.013110,175560,924
1,ANNDATS_next,AbsFundNews,"[-5,-1]",0.000131,0.000125,1.044311,0.002179,0.005527,175560,924
2,ACTDATS_next,FundNews,"[-5,-1]",0.000193,0.000131,1.480150,0.002990,0.008163,175750,925
3,ACTDATS_next,AbsFundNews,"[-5,-1]",0.000145,0.000134,1.083445,0.002438,0.006128,175750,925
4,MIN_ANN_ACT_next,FundNews,"[-5,-1]",0.000310,0.000137,2.267792,0.004783,0.013122,175560,924
5,MIN_ANN_ACT_next,AbsFundNews,"[-5,-1]",0.000131,0.000125,1.043899,0.002178,0.005524,175560,924
6,ANNDATS_next,FundNews,"[-5,0]",0.000346,0.000124,2.789017,0.005512,0.014660,175560,924
7,ANNDATS_next,FundNews,"[-10,-1]",0.000249,0.000116,2.145211,0.004403,0.010536,175560,924
8,ANNDATS_next,FundNews,"[-20,-1]",0.000219,0.000115,1.903429,0.004567,0.009251,175560,924
9,ANNDATS_next,FundNews,"[-1,-1]",0.000004,0.000140,0.027215,0.000048,0.000162,175560,924


## FF12 Sector-Predictor Relevance

In [6]:

ind=pd.read_csv(RAW/'industry_codes.csv').rename(columns={'PERMNO':'permno','HSICCD':'siccd'})
ind['ff12']=ind['siccd'].map(ff12_from_sic)
selection_ind=selection_df.merge(ind[['permno','ff12']],on='permno',how='left').dropna(subset=['ff12'])
sector_rows=[]
for outcome in ['FundNews','AbsFundNews']:
    fund=build_fund_news('anndats').merge(ind[['permno','ff12']],on='permno',how='left').dropna(subset=['ff12'])
    fund.attrs['date_name']='ANNDATS_next_ff12'
    fr=fundrel_by_group(fund,outcome,5,False,'ff12',MIN_EVENTS_SECTOR)
    panel=selection_ind.merge(fr,on=['ff12','predictor'],how='inner').dropna(subset=['FundRel_w'])
    res=second_stage(panel,'SelectionRate')
    sector_rows.append({'outcome':outcome,'window':'[-5,-1]','date_label':'ANNDATS_next','level':'FF12',**res,'fundrel_rows':len(fr),'n_sectors':fr.ff12.nunique()})
    print(f"FF12 {outcome}: beta={res['beta']:.6g}, t2w={res['t_twoway']:.2f}, N={res['n_obs']:,}, sectors={fr.ff12.nunique()}")
sector=pd.DataFrame(sector_rows); sector.to_csv(SECTOR_CSV,index=False)
sector


FF12 FundNews: beta=-0.000313567, t2w=-1.20, N=246,240, sectors=12
FF12 AbsFundNews: beta=0.000115896, t2w=0.48, N=246,240, sectors=12


,outcome,window,date_label,level,beta,se_firm,se_predictor,se_twoway,n_obs,n_firms,n_predictors,mean_y,sd_y,sd_x,t_twoway,std_effect,effect_pct_mean,fundrel_rows,n_sectors
0,FundNews,"[-5,-1]",ANNDATS_next,FF12,-0.000314,0.000115,0.000263,0.000262,246240,1296,190,0.019498,0.045363,0.734786,-1.198141,-0.005079,-0.016082,2280,12
1,AbsFundNews,"[-5,-1]",ANNDATS_next,FF12,0.000116,0.000107,0.000238,0.000243,246240,1296,190,0.019498,0.045363,0.956734,0.476875,0.002444,0.005944,2280,12


## Save and Display Summary

In [7]:

summary={
    'mean_selection_rate': float(mean_selection),
    'robustness_table': str(SUMMARY_CSV),
    'margin_table': str(MARGIN_CSV),
    'sector_table': str(SECTOR_CSV),
    'n_robust_specs': int(len(robust)),
}
SUMMARY_JSON.write_text(json.dumps(summary,indent=2))
print('Robustness table')
print(robust[['date_label','outcome','window','beta','se_firm','se_predictor','se_twoway','t_twoway','std_effect','effect_pct_mean','n_obs','n_firms']].to_string(index=False))
print('\nMargin tests')
print(margins[['margin','outcome','window','beta','se_twoway','t_twoway','std_effect','effect_pct_mean','n_obs','n_firms']].to_string(index=False))
print('\nFF12 sector tests')
print(sector[['outcome','window','beta','se_twoway','t_twoway','std_effect','effect_pct_mean','n_obs','n_firms','n_sectors']].to_string(index=False))


Robustness table
      date_label     outcome   window     beta  se_firm  se_predictor  se_twoway  t_twoway  std_effect  effect_pct_mean  n_obs  n_firms
    ANNDATS_next    FundNews  [-5,-1] 0.000310 0.000119      0.000133   0.000137  2.265996    0.004778         0.013110 175560      924
    ANNDATS_next AbsFundNews  [-5,-1] 0.000131 0.000117      0.000119   0.000125  1.044311    0.002179         0.005527 175560      924
    ACTDATS_next    FundNews  [-5,-1] 0.000193 0.000118      0.000127   0.000131  1.480150    0.002990         0.008163 175750      925
    ACTDATS_next AbsFundNews  [-5,-1] 0.000145 0.000120      0.000125   0.000134  1.083445    0.002438         0.006128 175750      925
MIN_ANN_ACT_next    FundNews  [-5,-1] 0.000310 0.000119      0.000133   0.000137  2.267792    0.004783         0.013122 175560      924
MIN_ANN_ACT_next AbsFundNews  [-5,-1] 0.000131 0.000117      0.000119   0.000125  1.043899    0.002178         0.005524 175560      924
    ANNDATS_next    FundNews   